# 04 - Bronze: IMF IMTS bilateral trade totals

This notebook replaces the UN Comtrade path for Week 2 bilateral total trade extraction.

UN Comtrade caused two operational problems for this project: Databricks Free Edition serverless could not resolve `comtradeapi.un.org`, and the local fallback depends on a subscription key and daily quota. IMF IMTS has the same country-partner-year total goods trade concept needed for partner dependency analysis, requires no API key, and can be called directly from Databricks.

Scope:

- 21 project reporters: CEMAC + ECOWAS
- All available partner rows per reporter
- Annual goods exports, FOB, USD
- Annual goods imports, CIF, USD
- Years 2010-2023

Important limitation: IMF IMTS gives total goods trade by partner. It does not give HS product-level rows. Product-level analysis will need a separate source later, such as CEPII BACI or a repaired Comtrade path.

In [ ]:
# Cell 1 - Configuration
import json
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from typing import Any, Dict, List

import requests
from pyspark.sql import Row
from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

BASE_DATA_URL = "https://api.imf.org/external/sdmx/2.1/data/IMTS"
STRUCTURE_URL = "https://api.imf.org/external/sdmx/2.1/dataflow/IMF.STA/IMTS/latest"

CEMAC = ["CMR", "CAF", "TCD", "COG", "GNQ", "GAB"]
ECOWAS = [
    "BEN", "BFA", "CPV", "CIV", "GMB", "GHA", "GIN",
    "GNB", "LBR", "MLI", "NER", "NGA", "SEN", "SLE", "TGO",
]
ALL_REPORTERS = CEMAC + ECOWAS

INDICATORS = {
    "XG_FOB_USD": {
        "indicator_name": "Exports of goods, FOB, US dollar",
        "flow_code": "X",
        "flow_desc": "Export",
        "valuation_basis": "FOB",
    },
    "MG_CIF_USD": {
        "indicator_name": "Imports of goods, CIF, US dollar",
        "flow_code": "M",
        "flow_desc": "Import",
        "valuation_basis": "CIF",
    },
}

START_YEAR = 2010
END_YEAR = 2023
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5
SLEEP_BETWEEN_CALLS_SECONDS = 0.5

print("Catalog: cemac_ecowas_aes_trade")
print("Schema: bronze")
print(f"Source: IMF IMTS ({BASE_DATA_URL})")
print(f"Reporters: {len(ALL_REPORTERS)} countries")
print(f"Indicators: {len(INDICATORS)}")
print(f"Years: {START_YEAR}-{END_YEAR}")
print(f"API calls planned: {len(ALL_REPORTERS) * len(INDICATORS)}")

In [ ]:
# Cell 2 - Helper functions
def local_name(tag: str) -> str:
    """Return an XML tag name without its namespace."""
    return tag.rsplit("}", 1)[-1]


def is_partner_aggregate(code: str) -> bool:
    """IMF uses G001/TX*/GX* codes for World, residual, or regional aggregates."""
    return code == "G001" or code.startswith("TX") or code.startswith("GX")


def fetch_country_lookup() -> Dict[str, str]:
    """Fetch IMF IMTS country and partner labels from the official codelist."""
    headers = {"Accept": "application/vnd.sdmx.structure+json;version=1.0.0"}
    response = requests.get(
        STRUCTURE_URL,
        params={"references": "all"},
        headers=headers,
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()

    for codelist in payload["data"].get("codelists", []):
        if codelist.get("id") == "CL_IMTS_COUNTRY":
            return {
                item["id"]: item.get("name", item["id"])
                for item in codelist.get("codes", [])
            }

    raise ValueError("CL_IMTS_COUNTRY codelist not found in IMF structure response")


def fetch_imts_xml(reporter_iso3: str, indicator_code: str) -> str:
    """Fetch one reporter-indicator slice from IMF IMTS.

    Dimension order is COUNTRY.INDICATOR.COUNTERPART_COUNTRY.FREQUENCY.
    The blank counterpart dimension requests all partners.
    """
    key = f"{reporter_iso3}.{indicator_code}..A"
    url = f"{BASE_DATA_URL}/{key}"
    params = {
        "startPeriod": str(START_YEAR),
        "endPeriod": str(END_YEAR),
    }
    headers = {"User-Agent": "cemac-ecowas-aes-trade-observatory/0.1"}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=180)
            if response.status_code in (429, 500, 502, 503, 504) and attempt < MAX_RETRIES:
                wait = RETRY_DELAY_SECONDS * attempt
                print(f"  HTTP {response.status_code}; retrying in {wait}s")
                time.sleep(wait)
                continue
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as exc:
            if attempt < MAX_RETRIES:
                wait = RETRY_DELAY_SECONDS * attempt
                print(f"  Attempt {attempt} failed for {key}: {exc}")
                print(f"  Retrying in {wait}s")
                time.sleep(wait)
            else:
                raise

    raise RuntimeError(f"Failed to fetch IMF IMTS key {key}")


def parse_imts_xml(
    xml_text: str,
    country_lookup: Dict[str, str],
    loaded_at: datetime,
) -> List[Row]:
    """Parse IMF structure-specific SDMX XML into Spark Rows."""
    root = ET.fromstring(xml_text)
    rows: List[Row] = []

    for series in root.iter():
        if local_name(series.tag) != "Series":
            continue

        series_attrs = dict(series.attrib)
        indicator_code = series_attrs.get("INDICATOR")
        indicator = INDICATORS.get(indicator_code)
        if indicator is None:
            continue

        reporter_iso3 = series_attrs.get("COUNTRY", "")
        partner_iso3 = series_attrs.get("COUNTERPART_COUNTRY", "")

        for obs in series:
            if local_name(obs.tag) != "Obs":
                continue

            value = obs.attrib.get("OBS_VALUE")
            rows.append(Row(
                reporter_iso3=reporter_iso3,
                reporter_name=country_lookup.get(reporter_iso3, reporter_iso3),
                partner_iso3=partner_iso3,
                partner_name=country_lookup.get(partner_iso3, partner_iso3),
                is_partner_aggregate=is_partner_aggregate(partner_iso3),
                year=int(obs.attrib["TIME_PERIOD"]),
                indicator_code=indicator_code,
                indicator_name=indicator["indicator_name"],
                flow_code=indicator["flow_code"],
                flow_desc=indicator["flow_desc"],
                valuation_basis=indicator["valuation_basis"],
                frequency=series_attrs.get("FREQUENCY"),
                methodology=series_attrs.get("METHODOLOGY"),
                overlap=series_attrs.get("OVERLAP"),
                derivation_type=obs.attrib.get("DERIVATION_TYPE"),
                trade_value_usd=float(value) if value not in (None, "") else None,
                source="imf_imts",
                loaded_at=loaded_at,
            ))

    return rows


print("Helper functions defined.")

In [ ]:
# Cell 3 - Fetch IMF country labels and validate reporter codes
country_lookup = fetch_country_lookup()
missing_reporters = [code for code in ALL_REPORTERS if code not in country_lookup]

if missing_reporters:
    raise ValueError(f"Missing reporter code(s) in IMF codelist: {missing_reporters}")

print(f"Loaded {len(country_lookup)} IMF country/partner labels")
print("Project reporters:")
for code in ALL_REPORTERS:
    print(f"  {code}: {country_lookup[code]}")

In [ ]:
# Cell 4 - Extract all reporters, all partners, both flows
all_rows = []
loaded_at = datetime.now(timezone.utc)
extraction_start = datetime.now(timezone.utc)

for reporter_iso3 in ALL_REPORTERS:
    print(f"Reporter {reporter_iso3} ({country_lookup[reporter_iso3]}):")

    for indicator_code, indicator in INDICATORS.items():
        xml_text = fetch_imts_xml(reporter_iso3, indicator_code)
        rows = parse_imts_xml(xml_text, country_lookup, loaded_at)
        all_rows.extend(rows)

        partner_count = len({row.partner_iso3 for row in rows if not row.is_partner_aggregate})
        print(
            f"  {indicator_code} {indicator['flow_desc']}: "
            f"{len(rows):,} rows, {partner_count:,} individual partners"
        )
        time.sleep(SLEEP_BETWEEN_CALLS_SECONDS)

elapsed = (datetime.now(timezone.utc) - extraction_start).total_seconds()
print("\n=== EXTRACTION COMPLETE ===")
print(f"Rows parsed: {len(all_rows):,}")
print(f"Elapsed seconds: {elapsed:.1f}")

In [ ]:
# Cell 5 - Build Spark DataFrame
if not all_rows:
    raise ValueError("No IMF IMTS rows were parsed. Check the extraction output above.")

df_raw = spark.createDataFrame(all_rows)

dedupe_cols = [
    "reporter_iso3",
    "partner_iso3",
    "year",
    "indicator_code",
    "valuation_basis",
]
df = df_raw.dropDuplicates(dedupe_cols)

raw_count = df_raw.count()
deduped_count = df.count()

print(f"Raw rows: {raw_count:,}")
print(f"Deduped rows: {deduped_count:,}")
print(f"Duplicates removed: {raw_count - deduped_count:,}")
print()
df.printSchema()
print()
df.show(10, truncate=False)

In [ ]:
# Cell 6 - Write to bronze.bilateral_trade_raw
spark.sql("DROP TABLE IF EXISTS bronze.bilateral_trade_raw")

(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("bronze.bilateral_trade_raw"))

print("Write complete: bronze.bilateral_trade_raw")

In [ ]:
# Cell 7 - Verify coverage and inspect Cameroon partner data
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT reporter_iso3) AS reporters,
        COUNT(DISTINCT partner_iso3) AS partners_all_codes,
        COUNT(DISTINCT CASE WHEN is_partner_aggregate = false THEN partner_iso3 END) AS individual_partners,
        COUNT(DISTINCT year) AS years,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        ROUND(SUM(CASE WHEN flow_code = 'X' THEN trade_value_usd END) / 1e12, 2) AS exports_trillions_usd,
        ROUND(SUM(CASE WHEN flow_code = 'M' THEN trade_value_usd END) / 1e12, 2) AS imports_trillions_usd
    FROM bronze.bilateral_trade_raw
""").show()

print("Per-reporter coverage:")
spark.sql("""
    SELECT
        reporter_iso3,
        reporter_name,
        COUNT(*) AS rows,
        COUNT(DISTINCT year) AS years_present,
        COUNT(DISTINCT CASE WHEN is_partner_aggregate = false THEN partner_iso3 END) AS individual_partners
    FROM bronze.bilateral_trade_raw
    GROUP BY reporter_iso3, reporter_name
    ORDER BY reporter_iso3
""").show(25, truncate=False)

print("Cameroon's top 10 export destinations in 2023:")
spark.sql("""
    SELECT
        partner_iso3,
        partner_name,
        ROUND(trade_value_usd / 1e9, 2) AS exports_billions_usd
    FROM bronze.bilateral_trade_raw
    WHERE reporter_iso3 = 'CMR'
      AND flow_code = 'X'
      AND year = 2023
      AND is_partner_aggregate = false
    ORDER BY trade_value_usd DESC
    LIMIT 10
""").show(truncate=False)

print("Cameroon's top 10 import origins in 2023:")
spark.sql("""
    SELECT
        partner_iso3,
        partner_name,
        ROUND(trade_value_usd / 1e9, 2) AS imports_billions_usd
    FROM bronze.bilateral_trade_raw
    WHERE reporter_iso3 = 'CMR'
      AND flow_code = 'M'
      AND year = 2023
      AND is_partner_aggregate = false
    ORDER BY trade_value_usd DESC
    LIMIT 10
""").show(truncate=False)